# **SVM NOTEBOOK**

In [12]:
# Import 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    accuracy_score, 
    precision_score, 
    recall_score, 
    f1_score, 
    roc_auc_score, 
    roc_curve
)

sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')

In [13]:
# Preprocesser-functions
def reshape_zones(df: pd.DataFrame) -> pd.DataFrame:
    zone_dfs = []
    for zone in ["se1", "se2", "se3", "se4"]:
        zone_df = df[
            [
                "timestamp",
                "timestamp_local",
                f"temp_{zone}_c",
                f"wind_{zone}_kmh",
                f"rain_{zone}_mm",
                f"price_{zone}_eur_mwh",
            ]
        ].copy()

        zone_df = zone_df.rename(
            columns={
                f"temp_{zone}_c": "temperature_c",
                f"wind_{zone}_kmh": "wind_speed_kmh",
                f"rain_{zone}_mm": "rain_mm",
                f"price_{zone}_eur_mwh": "spot_price_eur_mwh",
            }
        )
        zone_df["zone"] = zone.upper()
        zone_dfs.append(zone_df)

    return pd.concat(zone_dfs, ignore_index=True)


def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["timestamp_local"] = pd.to_datetime(df["timestamp_local"], utc=True).dt.tz_convert("Europe/Stockholm")
    df = df.sort_values(["zone", "timestamp"]).reset_index(drop=True)
    df = df.drop_duplicates(subset=["zone", "timestamp"], keep="first")
    return df


def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["hour"] = df["timestamp_local"].dt.hour
    df["day_of_week"] = df["timestamp_local"].dt.dayofweek
    df["month"] = df["timestamp_local"].dt.month
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    return df


def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.sort_values(["zone", "timestamp"]).reset_index(drop=True)
    df["price_lag_24"] = df.groupby("zone")["spot_price_eur_mwh"].shift(24)
    df["price_lag_48"] = df.groupby("zone")["spot_price_eur_mwh"].shift(48)
    df["price_lag_168"] = df.groupby("zone")["spot_price_eur_mwh"].shift(168)
    return df


def prepare_data(df: pd.DataFrame, include_lags: bool = True) -> pd.DataFrame:
    df = reshape_zones(df)
    df = clean_data(df)
    df = add_time_features(df)
    if include_lags:
        df = add_lag_features(df)
    return df

In [ ]:
# Read data and run pipeline
raw_df = pd.read_csv('../dataset/all_zones_complete_2025.csv')
df_processed = prepare_data(raw_df, include_lags=True)
df_processed = df_processed.dropna().reset_index(drop=True)

print(f"Bearbetad data form (alla zoner): {df_processed.shape}")
df_processed.head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/all_zones_complete_2025.csv'

In [ ]:
# Create target-variabel on all zones at same time
HOURS_TO_SELECT = 6  # De 6 billigaste timmarna per dygn och zon

# Skapa datumkolumn för gruppering
df_processed['date'] = df_processed['timestamp_local'].dt.date

# Beräkna pris-rank per zon och dygn
df_processed['price_rank'] = df_processed.groupby(['zone', 'date'])['spot_price_eur_mwh'].rank(method='min', ascending=True)
df_processed['optimal_timme'] = (df_processed['price_rank'] <= HOURS_TO_SELECT).astype(int)

# One-hot-encode zoner så att modellen vet vilken elpriszon som hanteras
df_processed = pd.get_dummies(df_processed, columns=['zone'], drop_first=False)

print("Total fördelning av målvariabeln för samtliga zoner:")
print(df_processed['optimal_timme'].value_counts())

In [ ]:
# Choose feature and split in Train / Val / Test
# Inkludera de one-hot-kodade zonkolumnerna bland features
feature_cols = [
    'temperature_c', 'wind_speed_kmh', 'rain_mm', 
    'hour', 'day_of_week', 'month', 'is_weekend',
    'price_lag_24', 'price_lag_48', 'price_lag_168',
    'zone_SE1', 'zone_SE2', 'zone_SE3', 'zone_SE4'
]

X = df_processed[feature_cols]
y = df_processed['optimal_timme']

# 70% Train, 15% Validation, 15% Test (stratifierat)
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.1764, random_state=42, stratify=y_train_val)

print(f"Träningsset: {X_train.shape[0]} rader")
print(f"Valideringsset: {X_val.shape[0]} rader")
print(f"Testset: {X_test.shape[0]} rader")

In [ ]:
# Scaling with StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Hyperparam-optimization with GridSearchCV
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.1],
    'kernel': ['rbf']
}

svm = SVC(class_weight='balanced', probability=True, random_state=42)

grid_search = GridSearchCV(
    estimator=svm,
    param_grid=param_grid,
    scoring='f1',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train_scaled, y_train)

print("\nBästa parametrar:")
print(grid_search.best_params_)

best_model = grid_search.best_estimator_

In [ ]:
# Evaluation on model
def evaluate_model(model, X_data, y_data, dataset_name="Test"):
    y_pred = model.predict(X_data)
    y_prob = model.predict_proba(X_data)[:, 1]
    
    print(f"--- Utvärdering på {dataset_name} ---")
    print(f"Accuracy:  {accuracy_score(y_data, y_pred):.4f}")
    print(f"Precision: {precision_score(y_data, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_data, y_pred):.4f}")
    print(f"F1-score:  {f1_score(y_data, y_pred):.4f}")
    print(f"ROC-AUC:   {roc_auc_score(y_data, y_prob):.4f}\n")
    
    print("Classification Report:")
    print(classification_report(y_data, y_pred))

evaluate_model(best_model, X_test_scaled, y_test, dataset_name="Testset")